# TomatoLab — Colab 학습 실습

이 노트북은 실제 PlantVillage 데이터를 가져와 **데이터 분할 → CNN → ResNet18 → 테스트 평가 → 웹 데모**까지 실행합니다.

**시작 전:** Colab 메뉴 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. GPU를 사용하지 않아도 실행되지만 학습이 느립니다.

> 연구용 이미지 분류 실험입니다. 실제 농작물의 질병 진단에 사용하면 안 됩니다.

In [ ]:
import os, subprocess, sys
repo_dir = '/content/TomatoLab'
if not os.path.isdir(repo_dir):
    subprocess.run(['git', 'clone', 'https://github.com/oyueo-mm/TomatoLab.git', repo_dir], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'kagglehub>=1.0.2'], check=True)
import torch
print('GPU 사용 가능:', torch.cuda.is_available())

## 1. 데이터 다운로드 및 클래스 폴더 탐색

KaggleHub로 공개 데이터셋을 내려받습니다. Kaggle 접근 권한 문제로 자동 다운로드가 실패한다면 [Kaggle 페이지](https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset)에서 직접 다운로드 후 압축을 풀고 아래 `dataset_root` 경로를 수동 지정하세요.

In [ ]:
import kagglehub
from pathlib import Path
dataset_root = Path(kagglehub.dataset_download('abdallahalidev/plantvillage-dataset'))
candidates = sorted({p.parent for p in dataset_root.rglob('Tomato___healthy') if p.is_dir()}, key=str)
valid_sources = [p for p in candidates if len([x for x in p.iterdir() if x.is_dir() and x.name.startswith('Tomato___')]) == 10]
if not valid_sources:
    raise FileNotFoundError(f'10개 Tomato___ 클래스 폴더가 있는 경로를 찾지 못했습니다: {dataset_root}')
source = next((p for p in valid_sources if 'color' in str(p).lower()), valid_sources[0])
print('선택한 컬러 데이터 경로:', source)

## 2. 정확 중복 제거 및 학습·검증·테스트 분할

클래스별 70/15/15, 시드 42를 사용합니다. 유사 이미지 수준의 누수는 여전히 가능하므로 실제 환경 이미지 평가가 후속 연구에 필요합니다.

In [ ]:
from pathlib import Path
from scripts.prepare_data import prepare_dataset
import json
output = Path('data/split')
if not (output / 'split_summary.json').exists():
    summary = prepare_dataset(source, output, seed=42)
else:
    summary = json.loads((output / 'split_summary.json').read_text(encoding='utf-8'))
print('완전 중복 제거:', summary['exact_duplicates_removed'])
for label, stats in summary['classes'].items():
    print(f"{label}: train={stats['train']}, val={stats['val']}, test={stats['test']}")

## 3. 기본 CNN 학습

검증 Macro F1이 개선될 때만 가중치를 저장합니다. 먼저 5 epoch로 동작을 확인하고 필요하면 10~20 epoch로 늘려보세요.

In [ ]:
subprocess.run([sys.executable, 'train.py', '--model', 'cnn', '--data-dir', 'data/split', '--epochs', '5'], check=True)

## 4. ResNet18 전이학습

ImageNet 사전학습 가중치 다운로드가 필요합니다. 기본 CNN과 같은 데이터 분할을 사용합니다.

In [ ]:
subprocess.run([sys.executable, 'train.py', '--model', 'resnet18', '--data-dir', 'data/split', '--epochs', '5'], check=True)

## 5. 테스트셋 평가 및 혼동행렬

검증셋은 모델 선택에, 테스트셋은 최종 보고에 사용합니다. 데이터가 부족한 클래스는 반드시 Recall을 따로 검토하세요.

In [ ]:
for name in ['cnn', 'resnet18']:
    subprocess.run([sys.executable, 'evaluate.py', '--checkpoint', f'artifacts/{name}/best.pt', '--data-dir', 'data/split'], check=True)
from IPython.display import display, Image
for name in ['cnn', 'resnet18']:
    metrics = json.loads(Path(f'artifacts/{name}/test_metrics.json').read_text(encoding='utf-8'))
    print(f"{name}: Accuracy={metrics['accuracy']:.4f}, Macro F1={metrics['macro_f1']:.4f}")
    display(Image(filename=f'artifacts/{name}/test_confusion_matrix.png'))

## 6. 학습 결과 저장

Colab 런타임 종료 시 가중치 파일이 사라질 수 있습니다. 학습이 끝나면 아래 셀로 체크포인트를 다운로드하세요. 이후 로컬에서 `python app.py --checkpoint ...`로 데모를 실행할 수 있습니다.

In [ ]:
from google.colab import files
files.download('artifacts/resnet18/best.pt')

## 7. ONNX 모델 내보내기 (웹 브라우저 추론용)

PyTorch의 `best.pt`는 GitHub Pages에서 직접 사용할 수 없습니다. 아래 셀은 학습된 ResNet18을 ONNX로 변환하고, PyTorch·ONNX 출력값을 검증한 뒤 브라우저에서 사용할 모델 파일을 내려받습니다. **모델 크기가 수십 MB이므로 다운로드 용량에 유의하세요.**


In [ ]:
import subprocess, sys
from pathlib import Path
from google.colab import files
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnx>=1.16', 'onnxruntime>=1.18'], check=True)
subprocess.run([sys.executable, 'scripts/export_onnx.py', '--checkpoint', 'artifacts/resnet18/best.pt', '--output', 'docs/models/resnet18.onnx'], check=True)
print('ONNX 크기:', round(Path('docs/models/resnet18.onnx').stat().st_size / (1024 * 1024), 1), 'MiB')
files.download('docs/models/resnet18.onnx')
